# M05C · Best Practices in Bioinformatics Programming & Secrets Management
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M05_Databases_and_Scraping/M05C_Best_Practices_API_Keys.ipynb)

⏱ ~60 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M05A (NCBI Entrez)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Audit**: Explain the governance policies of public biomedical APIs (NCBI, UniProt) and the operational purpose of API keys and contact email identification.
- **Secure**: Protect sensitive API keys and database credentials using `.env` files, `python-dotenv`, and `.gitignore` to eliminate credential leak risks.
- **Throttle**: Design client-side rate-limiting and query-batching architectures using `time.sleep()`, comma-separated ID lists, and NCBI WebEnv history servers.
- **Shield**: Construct resilient network wrappers implementing exponential backoff retries, explicit timeouts, and HTTP status code validation.
- **Refactor**: Transform fragile, insecure legacy bioinformatics scripts into modular, reproducible, production-grade workflows.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Querying NCBI Entrez using Biopython ([`M05A`](file:///modules/M05_Databases_and_Scraping/M05A_Entrez_GenBank_PubMed.ipynb)).
- Reading and writing files using `pathlib.Path` ([`M02A`](file:///modules/M02_File_Handling/M02A_Text_Files.ipynb)).
- Standard Python exception handling (`try` / `except` / `finally`).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M05C"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, re, logging, requests
from Bio import Entrez
from dotenv import load_dotenv

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 5C.1 Why Email & API Keys Matter: Policy & Governance

### 4.1.1 Theory & Biological Context

Public bioinformatics repositories (NCBI, EMBL-EBI, UniProt, RCSB PDB) are public goods maintained with taxpayer funding. They serve tens of millions of queries daily from researchers worldwide.

### NCBI Usage Policy & Fair Use:
1. **Mandatory Identification (`Entrez.email`)**:
   - Why? If a programming error causes a loop to fire 1,000 queries per second, NCBI's network firewalls will trigger an automated IP ban. If an email is supplied, NCBI staff will attempt to contact you first to resolve the bug before blacklisting your institution's subnet.
2. **Rate Limits & API Keys**:
   - **Without API Key**: Strict limit of **3 requests per second**.
   - **With API Key**: Increased quota of **10 requests per second**.
3. **Peak Hours Restriction**: NCBI requests that large batch jobs (>1,000 queries) run during off-peak hours (between 9:00 PM and 5:00 AM US Eastern Time, or weekends).

### 4.1.2 Worked Example

In [ ]:
# 5C.1 Worked Example: Dynamic Rate Limit Calculation
def calculate_minimum_delay(has_api_key: bool, safety_margin: float = 1.1) -> float:
    """Calculates required delay in seconds between requests to stay within quota."""
    # NCBI limits: 3 req/s without key, 10 req/s with key
    max_rate = 10.0 if has_api_key else 3.0
    safe_rate = max_rate / safety_margin
    delay_seconds = 1.0 / safe_rate
    return delay_seconds

delay_nokey = calculate_minimum_delay(has_api_key=False)
delay_key = calculate_minimum_delay(has_api_key=True)

print(f"Delay required WITHOUT API key: {delay_nokey:.3f} s (max {1.0/delay_nokey:.1f} req/s)")
print(f"Delay required WITH API key:    {delay_key:.3f} s (max {1.0/delay_key:.1f} req/s)")

### 4.1.3 ✏️ Try It: Email Validator

Write a function `validate_user_email(email: str) -> bool`:
- Checks that `email` contains a non-empty username, an `'@'` symbol, and a valid domain with at least one dot (e.g. `user@university.edu`).
- Returns `True` if valid, `False` otherwise.

In [ ]:
def validate_user_email(email: str) -> bool:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
assert validate_user_email("student@iiitd.ac.in")
assert validate_user_email("lab.pi@nih.gov")
assert not validate_user_email("invalid-email")
assert not validate_user_email("@missing_user.com")
print("Section 5C.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Passing dummy emails like 'a@b.com'**: NCBI validates domain MX records. Using fake addresses risks immediate IP throttling.
- **Hardcoding personal email in shared tutorial notebooks**: Always load email from environment variables (`os.environ.get('NCBI_EMAIL')`) so colleagues who clone your repo use their own credentials.

## 5C.2 Secrets Management: .env, python-dotenv & .gitignore

### 4.2.1 Theory & Biological Context

**Never commit API keys, database passwords, or private access tokens to version control.**

### The Threat Model:
Public GitHub repositories are continuously crawled by automated bots that scrape exposed credentials within seconds of a `git push`. A leaked API key can result in:
- Account suspension by NCBI or cloud providers.
- Exhaustion of institutional rate-limit quotas.
- Security vulnerabilities compromising linked compute accounts.

### The Standard Industry Solution:
1. Store credentials in a local text file named `.env`:
   ```
   NCBI_API_KEY=3a7b9f8e1234567890abcdef
   NCBI_EMAIL=researcher@iiitd.ac.in
   ```
2. **Add `.env` to `.gitignore`** so git permanently ignores it.
3. Commit a template file named `.env.example` with empty dummy values.
4. Load variables in Python using `python-dotenv`:
   ```python
   from dotenv import load_dotenv
   load_dotenv() # Reads .env into os.environ
   ```

### 4.2.2 Worked Example

In [ ]:
# 5C.2 Worked Example: Safe Credential Loading & Masking
# Create a temporary local .env in OUT_DIR for demonstration
demo_env_file = OUT_DIR / ".env.demo"
demo_env_file.write_text("NCBI_API_KEY=demo_key_998877665544\nNCBI_EMAIL=demo.user@iiitd.ac.in\n", encoding="utf-8")

# Load variables from specific file
load_dotenv(demo_env_file)

def get_masked_secret(var_name: str) -> str:
    """Fetches environment variable and masks all but first and last 2 characters."""
    val = os.environ.get(var_name, "")
    if not val:
        return "<NOT SET>"
    if len(val) <= 6:
        return "***"
    return f"{val[:2]}{'*' * (len(val) - 4)}{val[-2:]}"

print("NCBI_API_KEY status:", get_masked_secret("NCBI_API_KEY"))
print("NCBI_EMAIL status:  ", os.environ.get("NCBI_EMAIL"))

### 4.2.3 ✏️ Try It: API Key Masker

Write a function `mask_api_key(key: str) -> str`:
- If `key` is empty or `None`, return `"<EMPTY>"`.
- If `len(key) <= 4`, return `len(key) * "*"`.
- If `len(key) > 4`, keep first 2 characters, mask middle with `*`, and keep last 2 characters.
  Example: `"abcdef1234"` -> `"ab******34"`.

In [ ]:
def mask_api_key(key: str) -> str:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
assert mask_api_key("") == "<EMPTY>"
assert mask_api_key("12") == "**"
assert mask_api_key("12345678") == "12****78"
print("Section 5C.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Committing .env after forgetting .gitignore**: Once a key is committed to git history, it remains accessible in commit logs even if you delete the file in a subsequent commit!
- **Hardcoding default fallback keys**: Writing `api_key = os.environ.get('KEY', 'my_real_secret')` defeats the entire security boundary; the fallback must always be `None` or an empty string.

## 5C.3 Rate Limiting & Batching: Architectural Optimization

### 4.3.1 Theory & Biological Context

Network requests across the internet are expensive. Issuing individual HTTP requests in a naive loop for 5,000 accessions is inefficient and risks firewall throttling.

### Two Complementary Optimization Strategies:

#### 1. Batching (`id=1,2,3...`):
- Instead of making 100 requests for 100 accessions, combine IDs into comma-separated chunks:
  `id="NM_000546.6,NM_001301717.2,NM_000059.4"`
- Download 100 records in a single round-trip HTTP transaction!

#### 2. Client-Side Throttling:
- Enforce a minimum interval between requests:
  ```python
  time.sleep(0.35) # Guarantees < 3 requests per second
  ```
- Use high-resolution timing (`time.perf_counter()`) to compute actual elapsed time so pauses are only applied when needed.

### 4.3.2 Worked Example

In [ ]:
# 5C.3 Worked Example: Chunking Accession Lists for Batch Retrieval
def chunk_accessions(id_list: list[str], chunk_size: int = 50) -> list[str]:
    """Partitions a list of IDs into comma-separated batch strings."""
    batches = []
    for i in range(0, len(id_list), chunk_size):
        chunk = id_list[i : i + chunk_size]
        batches.append(",".join(chunk))
    return batches

sample_ids = [f"NM_{i:06d}.1" for i in range(1, 126)]
batches = chunk_accessions(sample_ids, chunk_size=50)

print(f"Total individual IDs: {len(sample_ids)}")
print(f"Batches created:      {len(batches)}")
print(f"Batch 1 (first 3 IDs): {batches[0].split(',')[:3]}...")
print(f"Batch 3 (count):       {len(batches[2].split(','))} IDs")

### 4.3.3 ✏️ Try It: Rate-Limited Batch Simulator

Write a function `simulate_rate_limited_batch(batches: list[str], max_rate_per_sec: float = 3.0) -> float`:
- Computes required minimum delay `1.0 / max_rate_per_sec`.
- Measures total simulated duration using a loop with `time.sleep(delay)`.
- (To keep tests fast, test with 3 tiny batches and `max_rate_per_sec=20.0`).
- Returns total elapsed seconds as a float.

In [ ]:
def simulate_rate_limited_batch(batches: list[str], max_rate_per_sec: float = 3.0) -> float:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
dur = simulate_rate_limited_batch(["b1", "b2", "b3"], max_rate_per_sec=50.0)
assert dur >= 3 * (1.0 / 50.0) * 0.9
print("Section 5C.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Batching > 500 IDs in a single GET URL**: Very long GET URLs (>2,048 characters) exceed web server header buffers and trigger HTTP 414 URI Too Long; chunk IDs in moderate batches (50-200).
- **Using time.sleep without measuring network latency**: Network calls take real time; pausing a fixed delay after a slow network response adds unnecessary idle time. Measure delta between timestamps for adaptive throttling.

## 5C.4 Robust Network Requests & Exponential Backoff

### 4.4.1 Theory & Biological Context

Distributed web services occasionally fail. Common biological API failure modes include:
- **HTTP 429 Too Many Requests**: You exceeded the rate limit.
- **HTTP 500 / 502 / 503 Server Error**: NCBI or UniProt servers are temporarily overloaded or undergoing maintenance.
- **Socket / DNS Timeout**: Network connection dropped.

### Resilient Architecture: Exponential Backoff with Jitter
Instead of crashing or immediately retrying in a tight loop (which amplifies server overload):
1. Catch connection and HTTP exceptions.
2. If error code is retryable (429, 500, 502, 503, 504), wait:
   $$t_{\text{wait}} = \text{base} \times 2^{\text{attempt}} + \text{jitter}$$
3. Exponentially scale delay ($1\text{s} \to 2\text{s} \to 4\text{s} \to 8\text{s}$).
4. Fail gracefully if maximum retry count is reached.

### 4.4.2 Worked Example

In [ ]:
# 5C.4 Worked Example: Robust Request Wrapper with Exponential Backoff
def robust_get_simulation(success_on_attempt: int = 2, max_retries: int = 3, base_delay: float = 0.05) -> dict:
    """Simulates transient failures and verifies exponential backoff retry logic."""
    attempt = 0
    while attempt < max_retries:
        attempt += 1
        print(f"Attempt {attempt}/{max_retries}...")
        if attempt >= success_on_attempt:
            print("✔ Request succeeded!")
            return {"status": 200, "attempts": attempt, "data": "Biological payload"}
        
        # Calculate exponential backoff delay
        wait_time = base_delay * (2 ** (attempt - 1))
        print(f"  Transient failure (HTTP 429). Backing off for {wait_time:.3f} s...")
        time.sleep(wait_time)
        
    raise RuntimeError(f"Request failed after {max_retries} attempts.")

res = robust_get_simulation(success_on_attempt=2)
print("Result:", res)

### 4.4.3 ✏️ Try It: Retryable HTTP Code Classifier

Write a function `is_retryable_http_code(status_code: int) -> bool`:
- Returns `True` if `status_code` indicates a transient failure: `429` (Too Many Requests), `500` (Internal Error), `502` (Bad Gateway), `503` (Service Unavailable), or `504` (Gateway Timeout).
- Returns `False` for permanent client errors like `400` (Bad Request), `401` (Unauthorized), `404` (Not Found), or `200` (OK).

In [ ]:
def is_retryable_http_code(status_code: int) -> bool:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
assert is_retryable_http_code(429)
assert is_retryable_http_code(503)
assert not is_retryable_http_code(404)
assert not is_retryable_http_code(200)
assert not is_retryable_http_code(400)
print("Section 5C.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Retrying in an infinite while loop without max_retries**: An infinite retry loop will hang your script permanently if an endpoint goes down for scheduled maintenance.
- **Omitting explicit timeouts on network calls**: Calling `requests.get(url)` without `timeout=10` can block indefinitely if the server drops the TCP socket without closing it.

## 5C.5 Reproducibility, Provenance & Checksums

### 4.5.1 Theory & Biological Context

In computational biology, reproducibility requires documenting **where**, **when**, and **how** data was acquired.

### Data Provenance Checklist:
1. **Accession Versioning**: Always cite `NM_000546.6` instead of `NM_000546`.
2. **Metadata Sidecars**: Whenever saving a raw dataset, write an accompanying `.meta.json` recording:
   - Source database URL.
   - Exact query string and parameters.
   - ISO 8601 UTC timestamp.
   - SHA-256 cryptographic checksum of raw bytes.
3. **Structured Logging**: Replace unstructured `print()` statements with Python's standard `logging` module.

### 4.5.2 Worked Example

In [ ]:
# 5C.5 Worked Example: Generating Data Provenance Metadata
import hashlib, json
from datetime import datetime, timezone

def generate_provenance_record(file_path: Path, source_url: str, query: str) -> dict:
    """Computes SHA-256 hash and provenance metadata for a downloaded file."""
    data = file_path.read_bytes()
    sha256_hash = hashlib.sha256(data).hexdigest()
    
    metadata = {
        "file_name": file_path.name,
        "byte_size": len(data),
        "sha256": sha256_hash,
        "source_url": source_url,
        "query": query,
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    }
    
    meta_path = file_path.with_suffix(file_path.suffix + ".meta.json")
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata

# Test on local fasta
demo_fasta = DATA_DIR / "sequences" / "tp53_human_mrna.fasta"
meta = generate_provenance_record(demo_fasta, "https://eutils.ncbi.nlm.nih.gov/", "NM_000546.6")
print(f"Provenance recorded: SHA256={meta['sha256'][:16]}... Date={meta['timestamp_utc'][:10]}")

### 4.5.3 ✏️ Try It: File Integrity Checker

Write a function `verify_file_integrity(file_path: Path, expected_sha256: str) -> bool`:
- Calculates SHA-256 hash of `file_path`.
- Returns `True` if it matches `expected_sha256` (case-insensitive), else `False`.

In [ ]:
def verify_file_integrity(file_path: Path, expected_sha256: str) -> bool:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
assert verify_file_integrity(demo_fasta, meta["sha256"])
assert not verify_file_integrity(demo_fasta, "wronghash1234")
print("Section 5C.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Relying on file modification dates for versioning**: File timestamps change whenever a file is copied or moved across filesystems; use cryptographic hashes for true integrity checks.
- **Omitting timezones from timestamps**: Always use UTC timestamps (`datetime.now(timezone.utc)`) to avoid ambiguity between global collaborating laboratories.

### Practice Exercise 5C.1 (★) — RefSeq Accession Version Enforcer

**Problem**: Write a function `enforce_accession_version(accession: str) -> tuple[bool, str]`:
- Takes an accession string (e.g. `"NM_000546.6"` or `"NM_000546"`).
- If accession has a valid version suffix (e.g. `".6"`), return `(True, "Valid versioned accession")`.
- If accession lacks a version suffix, return `(False, "Missing accession version suffix (.N)")`.
- If accession does not match standard RefSeq prefix syntax, return `(False, "Invalid RefSeq accession syntax")`.


- **Input**: Accession string
- **Expected Output**: Tuple of (bool, explanation_string)

In [ ]:
def enforce_accession_version(accession: str) -> tuple[bool, str]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
ok, msg = enforce_accession_version("NM_000546.6")
assert ok and "Valid" in msg

ok, msg = enforce_accession_version("NM_000546")
assert not ok and "Missing" in msg

ok, msg = enforce_accession_version("invalid_acc")
assert not ok and "Invalid" in msg
print("Ex 5C.1 passed ✔")

### Practice Exercise 5C.2 (★) — Safe Environment Secret Loader with Fallback

**Problem**: Write a function `load_bio_credentials() -> dict[str, str]`:
- Loads environment variables from `.env` (if present) using `python-dotenv`.
- Retrieves `'NCBI_EMAIL'` and `'NCBI_API_KEY'`.
- If `'NCBI_EMAIL'` is not set, defaults to `'anonymous@student.edu'`.
- If `'NCBI_API_KEY'` is not set, defaults to `''`.
- Returns dictionary `{'email': email, 'api_key': api_key, 'masked_key': masked_key}` where `masked_key` uses `mask_api_key`.


- **Input**: None (reads environment)
- **Expected Output**: Dictionary with keys 'email', 'api_key', 'masked_key'

In [ ]:
def load_bio_credentials() -> dict[str, str]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
creds = load_bio_credentials()
assert "email" in creds and "@" in creds["email"]
assert "masked_key" in creds
print("Ex 5C.2 passed ✔")

### Practice Exercise 5C.3 (★★) — Adaptive Rate Limiter Class

**Problem**: Implement a class `RateLimiter`:
- Initialized with `calls_per_second: float = 3.0`.
- Has method `pause()`: Computes elapsed time since the previous call. If elapsed time is less than `1.0 / calls_per_second`, calls `time.sleep()` for the remaining difference.
- Records current timestamp as previous call.


- **Input**: Calls per second limit
- **Expected Output**: RateLimiter class enforcing minimum separation intervals

In [ ]:
class RateLimiter:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
limiter = RateLimiter(calls_per_second=20.0) # 0.05s interval
t0 = time.perf_counter()
limiter.pause()
limiter.pause()
limiter.pause()
t_total = time.perf_counter() - t0
assert t_total >= 0.09, f"Elapsed {t_total:.3f} s should be at least ~0.10 s"
print("Ex 5C.3 passed ✔")

### Practice Exercise 5C.4 (★★) — Exponential Backoff HTTP Fetcher with Status Check

**Problem**: Write a function `fetch_with_backoff(url: str, params: dict, max_retries: int = 3, base_wait: float = 0.05) -> requests.Response`:
- Uses `requests.get(url, params=params, timeout=5)`.
- If response status code is 200, returns the response.
- If status code is retryable (429, 500, 502, 503), sleeps `base_wait * (2 ** attempt)` and retries.
- If status code is non-retryable (e.g. 404), raises `requests.HTTPError`.
- If max_retries is exhausted, raises `RuntimeError("Max retries exceeded")`.


- **Input**: URL string, params dict, max_retries int
- **Expected Output**: Successful requests.Response or appropriate exception

In [ ]:
def fetch_with_backoff(url: str, params: dict, max_retries: int = 3, base_wait: float = 0.05) -> requests.Response:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
# Test against UniProt API endpoint
try:
    r = fetch_with_backoff("https://rest.uniprot.org/uniprotkb/search",
                           {"query": "gene:TP53 AND reviewed:true", "size": 1, "format": "tsv"})
    assert r.status_code == 200
except Exception as e:
    print(f"Network offline test passed ({e})")
print("Ex 5C.4 passed ✔")

### Practice Exercise 5C.5 (★★★) — Refactoring a Fragile Legacy Script into Production Code

**Problem**: Examine this brittle, insecure legacy script:
```python
# BAD SCRIPT: DO NOT REPLICATE
import os
from Bio import Entrez
Entrez.email = "my_personal_email@gmail.com" # Hardcoded personal email
Entrez.api_key = "12345_insecure_key"         # Hardcoded secret
ids = ["NM_000546", "NM_001301717"]          # Missing versions
for i in ids:
    h = Entrez.efetch(db="nucleotide", id=i, rettype="fasta") # No sleep, no try/except
    with open("C:/Users/Prateek/Desktop/" + i + ".fa", "w") as f: # Hardcoded OS path
        f.write(h.read())
```

Write a refactored production function `download_refseq_batch(accessions: list[str], dest_dir: Path) -> list[Path]`:
1. Validates accessions using `enforce_accession_version` (or rejects if invalid).
2. Loads credentials from environment.
3. Implements rate limiting and local file caching (does not re-download if file exists).
4. Wraps Entrez fetch in `try/except` with fallback to local mock data.
5. Saves FASTA files to `dest_dir` using `pathlib.Path`.
6. Returns list of saved `Path` objects.


- **Input**: List of accessions and destination Path
- **Expected Output**: List of Path objects for verified files

In [ ]:
def download_refseq_batch(accessions: list[str], dest_dir: Path) -> list[Path]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
dest = OUT_DIR / "refseq_batch"
downloaded = download_refseq_batch(["NM_000546.6", "NM_001301717.2"], dest)
assert len(downloaded) == 2
assert all(p.exists() and p.stat().st_size > 0 for p in downloaded)
print("Ex 5C.5 passed ✔")

## 6. Summary & Cheat Sheet

Bioinformatics software engineering requires rigorous attention to security, rate limiting, and reproducibility. By decoupling secrets into `.env`, throttling queries, adopting retry backoff, and recording cryptographic data provenance, your pipelines become robust and publication-ready.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Secrets Storage | .env file + python-dotenv | Credential security | Add .env to .gitignore |
| Rate Limiting | time.sleep(1.0 / quota) | Server etiquette | 3 req/s without key, 10 req/s with key |
| Batching | id='acc1,acc2,acc3' | Network optimization | Batch 50-200 IDs per request |
| Exponential Backoff | base * (2 ** attempt) | Fault tolerance | Retry 429 and 503 errors only |
| Timeouts | requests.get(url, timeout=10) | Hanging socket prevention | Always specify timeout |
| Provenance | hashlib.sha256() + UTC timestamp | Scientific reproducibility | Save sidecar metadata |

## 7. Further Reading & Resources

- [The Twelve-Factor App: Config](https://12factor.net/config): Foundational principles for storing config and credentials in environment variables.
- [NCBI API Keys and Usage Policies](https://ncbiinsights.ncbi.nlm.nih.gov/2017/11/02/new-api-keys-for-the-e-utilities/): Official guidance on obtaining and configuring NCBI API keys.
- [Python Requests Documentation: Timeouts and Retries](https://requests.readthedocs.io/): Best practices for HTTP error handling.